In [16]:
import requests
import pandas as pd
from io import StringIO
import time
import random
from IPython.display import display

# ── Filter config ──────────────────────────────────────────────
PROPERTY_TYPE_MAP = {
    "house":     "house",
    "condo":     "condo",
    "townhouse": "townhouse",
    "land":      "land",
    "other":     "other",
}

UIPT_MAP = {
    "house":     "1",
    "condo":     "2",
    "townhouse": "3",
    "land":      "6",
    "other":     "8",
}

def format_price(price: int) -> str:
    if price >= 1_000_000:
        val = price / 1_000_000
        formatted = int(val) if val == int(val) else round(val, 1)
        return f"{formatted}M"
    elif price >= 1_000:
        val = price / 1_000
        formatted = int(val) if val == int(val) else round(val, 1)
        return f"{formatted}K"
    return str(price)


def build_redfin_filter_url(
    region_id, state, city,
    property_types=None, min_price=None, max_price=None,
    min_beds=None, max_beds=None, min_baths=None, max_baths=None,
    min_year_built=None, max_year_built=None,
    min_sqft=None, max_sqft=None,
    exclude_age_restricted=True, include_open_houses=True,
) -> str:
    filters = []
    if property_types:
        valid = [PROPERTY_TYPE_MAP[t] for t in property_types if t in PROPERTY_TYPE_MAP]
        if valid:
            filters.append("property-type=" + "+".join(valid))
    if min_price:   filters.append(f"min-price={format_price(min_price)}")
    if max_price:   filters.append(f"max-price={format_price(max_price)}")
    if min_beds:    filters.append(f"min-beds={min_beds}")
    if max_beds:    filters.append(f"max-beds={max_beds}")
    if min_baths:   filters.append(f"min-baths={min_baths}")
    if max_baths:   filters.append(f"max-baths={max_baths}")
    if min_year_built: filters.append(f"min-year-built={min_year_built}")
    if max_year_built: filters.append(f"max-year-built={max_year_built}")
    if min_sqft:    filters.append(f"min-sqft={min_sqft}")
    if max_sqft:    filters.append(f"max-sqft={max_sqft}")
    if exclude_age_restricted: filters.append("exclude-age-restricted")
    if include_open_houses:    filters.append("open-house-time=this-weekend")

    base = f"https://www.redfin.com/city/{region_id}/{state}/{city}"
    return f"{base}/filter/{','.join(filters)}" if filters else base


def scrape_open_houses(
    city, state, region_id,
    property_types=None, min_price=None, max_price=None,
    min_beds=None, max_beds=None, min_baths=None, max_baths=None,
    min_year_built=None, max_year_built=None,
    min_sqft=None, max_sqft=None,
    exclude_age_restricted=True,
) -> list[dict]:

    listing_url = build_redfin_filter_url(
        region_id=region_id, state=state, city=city,
        property_types=property_types,
        min_price=min_price, max_price=max_price,
        min_beds=min_beds, max_beds=max_beds,
        min_baths=min_baths, max_baths=max_baths,
        min_year_built=min_year_built, max_year_built=max_year_built,
        min_sqft=min_sqft, max_sqft=max_sqft,
        exclude_age_restricted=exclude_age_restricted,
        include_open_houses=True,
    )
    print(f"Fetching: {listing_url}")

    session = requests.Session()
    headers_browser = {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.9",
        "Accept-Encoding": "gzip, deflate, br",
        "Connection": "keep-alive",
    }
    try:
        session.get(listing_url, headers=headers_browser, timeout=15)
        time.sleep(random.uniform(2, 4))
    except Exception as e:
        print(f"Warning: session page failed: {e}")

    uipt_codes = (
        ",".join(UIPT_MAP[t] for t in property_types if t in UIPT_MAP)
        if property_types else "1,2,3"
    )
    csv_url = (
        f"https://www.redfin.com/stingray/api/gis-csv"
        f"?al=1"
        f"&market={state.lower()}"
        f"&region_id={region_id}"
        f"&region_type=6"
        f"&sf=1,2,3,5,6,7"
        f"&num_homes=350"
        f"&uipt={uipt_codes}"
        f"&open_house_time=this-weekend"
    )
    headers_csv = {
        **headers_browser,
        "Referer": listing_url,
        "Sec-Fetch-Site": "same-origin",
    }
    try:
        r = session.get(csv_url, headers=headers_csv, timeout=15)
        print(f"Status: {r.status_code} | Content-Type: {r.headers.get('Content-Type')}")
        r.raise_for_status()
    except Exception as e:
        print(f"Request failed: {e}")
        return []

    # Redfin prepends a disclaimer line before the CSV header — skip it
    raw = r.text
    print("First 300 chars of response:\n", raw[:300])
    try:
        # Try skipping the first row (disclaimer); if that produces bad columns, try without
        df = pd.read_csv(StringIO(raw), skiprows=1)
        if df.columns[0].startswith("SALE TYPE") or "ADDRESS" in df.columns:
            pass  # good parse
        else:
            df = pd.read_csv(StringIO(raw))
    except Exception as e:
        print(f"CSV parse error: {e}\nRaw: {raw[:300]}")
        return []

    print(f"Columns: {df.columns.tolist()}")
    print(f"Rows before filtering: {len(df)}")

    # Clean numeric columns (Redfin uses ALL-CAPS headers)
    price_col  = "PRICE"
    sqft_col   = "SQUARE FEET"
    beds_col   = "BEDS"
    baths_col  = "BATHS"
    yr_col     = "YEAR BUILT"

    if price_col in df.columns:
        df[price_col] = pd.to_numeric(df[price_col].astype(str).str.replace(r'[\$,]', '', regex=True), errors='coerce')
    if sqft_col in df.columns:
        df[sqft_col]  = pd.to_numeric(df[sqft_col].astype(str).str.replace(r'[,]', '', regex=True), errors='coerce')

    # ── Top 5 properties before filtering ─────────────────────
    preview_cols = [c for c in ["ADDRESS", price_col, beds_col, baths_col, sqft_col, yr_col] if c in df.columns]
    print("\nTop 5 properties before filtering:")
    display(df[preview_cols].head(5))

    if min_beds  and beds_col  in df.columns: df = df[df[beds_col]  >= min_beds]
    if max_beds  and beds_col  in df.columns: df = df[df[beds_col]  <= max_beds]
    if min_price and price_col in df.columns: df = df[df[price_col] >= min_price]
    if max_price and price_col in df.columns: df = df[df[price_col] <= max_price]
    if min_year_built and yr_col in df.columns: df = df[df[yr_col]  >= min_year_built]
    if min_sqft  and sqft_col  in df.columns: df = df[df[sqft_col]  >= min_sqft]
    if max_sqft  and sqft_col  in df.columns: df = df[df[sqft_col]  <= max_sqft]

    print(f"Rows after filtering: {len(df)}")

    # Redfin CSV URL column name (long)
    url_col = next((c for c in df.columns if c.startswith("URL")), "")

    results = []
    for _, row in df.iterrows():
        results.append({
            "address":         row.get("ADDRESS", ""),
            "city":            row.get("CITY", city),
            "state":           row.get("STATE OR PROVINCE", state),
            "zip":             row.get("ZIP OR POSTAL CODE", ""),
            "price":           row.get(price_col, None),
            "beds":            row.get(beds_col, None),
            "baths":           row.get(baths_col, None),
            "sqft":            row.get(sqft_col, None),
            "year_built":      row.get(yr_col, None),
            "open_house_time": row.get("NEXT OPEN HOUSE START TIME", None),
            "latitude":        row.get("LATITUDE", None),
            "longitude":       row.get("LONGITUDE", None),
            "url":             row.get(url_col, "") if url_col else "",
        })

    return results


# ── Test ───────────────────────────────────────────────────────
houses = scrape_open_houses(
    city="Redmond",
    state="WA",
    region_id="14913",
    property_types=["house", "condo", "townhouse"],
    max_price=1_500_000,
    min_beds=3,
    min_baths=1.5,
    min_year_built=1980,
    exclude_age_restricted=True,
)

print(f"\nFound {len(houses)} open houses")
for h in houses[:3]:
    print(h)


Fetching: https://www.redfin.com/city/14913/WA/Redmond/filter/property-type=house+condo+townhouse,max-price=1.5M,min-beds=3,min-baths=1.5,min-year-built=1980,exclude-age-restricted,open-house-time=this-weekend
Status: 200 | Content-Type: text/csv;charset=UTF-8
First 300 chars of response:
 SALE TYPE,SOLD DATE,PROPERTY TYPE,ADDRESS,CITY,STATE OR PROVINCE,ZIP OR POSTAL CODE,PRICE,BEDS,BATHS,LOCATION,SQUARE FEET,LOT SIZE,YEAR BUILT,DAYS ON MARKET,$/SQUARE FEET,HOA/MONTH,STATUS,NEXT OPEN HOUSE START TIME,NEXT OPEN HOUSE END TIME,URL (SEE https://www.redfin.com/buy-a-home/comparative-marke
Columns: ['SALE TYPE', 'SOLD DATE', 'PROPERTY TYPE', 'ADDRESS', 'CITY', 'STATE OR PROVINCE', 'ZIP OR POSTAL CODE', 'PRICE', 'BEDS', 'BATHS', 'LOCATION', 'SQUARE FEET', 'LOT SIZE', 'YEAR BUILT', 'DAYS ON MARKET', '$/SQUARE FEET', 'HOA/MONTH', 'STATUS', 'NEXT OPEN HOUSE START TIME', 'NEXT OPEN HOUSE END TIME', 'URL (SEE https://www.redfin.com/buy-a-home/comparative-market-analysis FOR INFO ON PRICING)', 'SO

,ADDRESS,PRICE,BEDS,BATHS,SQUARE FEET,YEAR BUILT
0,NaN,NaN,NaN,NaN,NaN,NaN
1,12255 138th Pl NE Unit 17-102,1314900.0,4.0,3.5,2188.0,2026.0
2,12255 138th Pl NE Unit 17-101,1397000.0,4.0,3.5,2281.0,2026.0
3,12217 137th Pl NE Unit 27-104,899900.0,2.0,2.5,1539.0,2026.0
4,10534 135th Pl NE #35,2417995.0,6.0,2.5,3422.0,2026.0


Rows after filtering: 3

Found 3 open houses
{'address': '12255 138th Pl NE Unit 17-102', 'city': 'Redmond', 'state': 'WA', 'zip': 98052.0, 'price': 1314900.0, 'beds': 4.0, 'baths': 3.5, 'sqft': 2188.0, 'year_built': 2026.0, 'open_house_time': nan, 'latitude': 47.7101754, 'longitude': -122.157705, 'url': 'https://www.redfin.com/WA/Redmond/12255-138th-Pl-NE-98052/unit-17-102/home/201708205'}
{'address': '12255 138th Pl NE Unit 17-101', 'city': 'Redmond', 'state': 'WA', 'zip': 98052.0, 'price': 1397000.0, 'beds': 4.0, 'baths': 3.5, 'sqft': 2281.0, 'year_built': 2026.0, 'open_house_time': nan, 'latitude': 47.7101754, 'longitude': -122.157705, 'url': 'https://www.redfin.com/WA/Redmond/12255-138th-Pl-NE-98052/unit-17-101/home/201708198'}
{'address': '12201 137th Pl NE Unit 25-102', 'city': 'Redmond', 'state': 'WA', 'zip': 98052.0, 'price': 1199900.0, 'beds': 4.0, 'baths': 3.5, 'sqft': 2053.0, 'year_built': 2026.0, 'open_house_time': nan, 'latitude': 47.7095374, 'longitude': -122.15815, 'url